# Edmonton Residential Waste Cart Rollout Analysis

### A portfolio case study using City of Edmonton Open Data and 2021 Census data

**Business question**

How did Edmonton's residential cart rollout vary across time and neighbourhoods, and how does the picture change after accounting for neighbourhood size, occupied dwellings, and recent population growth?

**Why this matters**

Raw cart counts can overstate the importance of large neighbourhoods. This analysis moves from absolute counts to more decision-useful measures, especially **carts with 2021 install dates per 1,000 occupied dwellings**.

**Tools demonstrated**

Python · pandas · REST/Socrata API · GeoPandas · Matplotlib · Census normalization · Data-quality auditing · Correlation · Segmentation · Outlier analysis

## Executive summary

This notebook is structured like an analyst deliverable rather than a coding exercise.

The analysis is designed to establish five things:

- when the main cart rollout activity occurred;
- which neighbourhoods were most affected;
- whether rollout timing was spatially structured;
- whether population or occupied dwellings provide a better denominator;
- whether rapid neighbourhood growth helps explain unusually high rollout intensity.

> **Important:** `Cart Counts` should be treated as a snapshot of active carts grouped by installation date, not as a complete historical transaction log of every cart installation and removal.

## 1. Setup and reusable API helper

The notebook queries Edmonton Open Data directly. Aggregation is performed on the server where possible, so the full Cart Counts dataset does not need to be downloaded.

In [ ]:
import numpy as np
import pandas as pd
import requests
from io import StringIO

import matplotlib.pyplot as plt
import geopandas as gpd
from shapely import wkt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

CART_URL = "https://data.edmonton.ca/resource/q2sn-xztr.csv"
POP_2019_URL = "https://data.edmonton.ca/resource/a6zx-dzqn.csv"
POP_2021_URL = "https://data.edmonton.ca/resource/eg3i-f4bj.csv"
CENSUS_2021_SHORT_URL = "https://data.edmonton.ca/resource/rw8s-vqrs.csv"

def api_query(url, params=None):
    response = requests.get(url, params=params or {}, timeout=60)
    response.raise_for_status()
    return pd.read_csv(StringIO(response.text))

MONTH_NAMES = {
    1: "January", 2: "February", 3: "March", 4: "April",
    5: "May", 6: "June", 7: "July", 8: "August",
    9: "September", 10: "October", 11: "November", 12: "December"
}

## 2. Data-quality audit

Before interpreting patterns, verify the actual date range, record count, product categories, and annual distribution.

This is important because metadata descriptions and observed values do not always align perfectly.

In [ ]:
date_range = api_query(
    CART_URL,
    {
        "$select": """
            min(install_date) as earliest_install,
            max(install_date) as latest_install,
            count(*) as rows
        """
    }
)

date_range

In [ ]:
products = api_query(
    CART_URL,
    {
        "$select": """
            product_description,
            sum(number_of_carts) as carts,
            count(*) as records
        """,
        "$group": "product_description",
        "$order": "sum(number_of_carts) DESC"
    }
)

products["carts"] = pd.to_numeric(products["carts"], errors="coerce")
products["records"] = pd.to_numeric(products["records"], errors="coerce")

products

In [ ]:
yearly = api_query(
    CART_URL,
    {
        "$select": """
            date_extract_y(install_date) as year,
            sum(number_of_carts) as carts_installed,
            count(*) as records
        """,
        "$group": "date_extract_y(install_date)",
        "$order": "date_extract_y(install_date)"
    }
)

for col in ["year", "carts_installed", "records"]:
    yearly[col] = pd.to_numeric(yearly[col], errors="coerce")

yearly["share_pct"] = yearly["carts_installed"] / yearly["carts_installed"].sum() * 100

yearly

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(yearly["year"].astype("Int64").astype(str), yearly["carts_installed"])
ax.set_title("Active carts represented in the dataset by installation year")
ax.set_xlabel("Installation year")
ax.set_ylabel("Number of carts")
plt.tight_layout()
plt.show()

### Data-quality takeaway

If the observed install-date range extends outside the period emphasized in the dataset description, document that explicitly. Do not silently trim those records.

The annual distribution determines the analytical focus. In the working snapshot used to build this case, **2021 clearly dominates the installation-date distribution**.

## 3. 2021 rollout timeline

The next step is to determine whether 2021 activity was evenly distributed or concentrated in a specific rollout window.

In [ ]:
monthly_2021 = api_query(
    CART_URL,
    {
        "$select": """
            date_extract_m(install_date) as month,
            sum(number_of_carts) as carts_installed,
            count(*) as records
        """,
        "$where": """
            install_date >= '2021-01-01T00:00:00'
            AND install_date < '2022-01-01T00:00:00'
        """,
        "$group": "date_extract_m(install_date)",
        "$order": "date_extract_m(install_date)"
    }
)

monthly_2021["month"] = pd.to_numeric(monthly_2021["month"], errors="coerce").astype("Int64")
monthly_2021["carts_installed"] = pd.to_numeric(monthly_2021["carts_installed"], errors="coerce")
monthly_2021["records"] = pd.to_numeric(monthly_2021["records"], errors="coerce")
monthly_2021["month_name"] = monthly_2021["month"].map(MONTH_NAMES)

monthly_2021

In [ ]:
total_2021 = monthly_2021["carts_installed"].sum()
jan_aug = monthly_2021.loc[
    monthly_2021["month"].between(1, 8),
    "carts_installed"
].sum()

print(f"Total carts with 2021 install dates: {total_2021:,.0f}")
print(f"January-August share: {jan_aug / total_2021:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(monthly_2021["month_name"], monthly_2021["carts_installed"], marker="o")
ax.set_title("2021 cart rollout timeline")
ax.set_xlabel("Month")
ax.set_ylabel("Carts with 2021 installation dates")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

### Interpretation

A steep decline after the main rollout window suggests that the 2021 distribution was highly concentrated in time.

Missing months mean **no records were observed with those installation dates**. They should not automatically be interpreted as proof that no related operational activity occurred.

## 4. Product mix

Check whether the monthly rollout pattern changes by cart type. This helps avoid attributing a shift to geography when it may actually reflect a change in product mix.

In [ ]:
monthly_products_2021 = api_query(
    CART_URL,
    {
        "$select": """
            date_extract_m(install_date) as month,
            product_description,
            sum(number_of_carts) as carts_installed
        """,
        "$where": """
            install_date >= '2021-01-01T00:00:00'
            AND install_date < '2022-01-01T00:00:00'
        """,
        "$group": """
            date_extract_m(install_date),
            product_description
        """,
        "$order": """
            date_extract_m(install_date),
            sum(number_of_carts) DESC
        """,
        "$limit": 50000
    }
)

monthly_products_2021["month"] = pd.to_numeric(monthly_products_2021["month"], errors="coerce")
monthly_products_2021["carts_installed"] = pd.to_numeric(
    monthly_products_2021["carts_installed"], errors="coerce"
)

product_pivot = (
    monthly_products_2021
    .pivot(index="month", columns="product_description", values="carts_installed")
    .fillna(0)
)

product_share = product_pivot.div(product_pivot.sum(axis=1), axis=0) * 100

product_pivot

In [ ]:
product_share.round(1)

## 5. Neighbourhood-level rollout

This section identifies the strongest neighbourhood activity by month and the peak rollout month for each neighbourhood.

In [ ]:
monthly_neighbourhoods_2021 = api_query(
    CART_URL,
    {
        "$select": """
            date_extract_m(install_date) as month,
            neighbourhood_number,
            neighbourhood_name,
            sum(number_of_carts) as carts_installed
        """,
        "$where": """
            install_date >= '2021-01-01T00:00:00'
            AND install_date < '2022-01-01T00:00:00'
        """,
        "$group": """
            date_extract_m(install_date),
            neighbourhood_number,
            neighbourhood_name
        """,
        "$order": """
            date_extract_m(install_date),
            sum(number_of_carts) DESC
        """,
        "$limit": 50000
    }
)

monthly_neighbourhoods_2021["month"] = pd.to_numeric(
    monthly_neighbourhoods_2021["month"], errors="coerce"
)
monthly_neighbourhoods_2021["carts_installed"] = pd.to_numeric(
    monthly_neighbourhoods_2021["carts_installed"], errors="coerce"
)

top5_monthly = (
    monthly_neighbourhoods_2021
    .sort_values(["month", "carts_installed"], ascending=[True, False])
    .groupby("month", group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

top5_monthly

In [ ]:
peak_month = (
    monthly_neighbourhoods_2021
    .sort_values(["neighbourhood_name", "carts_installed"], ascending=[True, False])
    .groupby("neighbourhood_name", group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

peak_month["peak_month_name"] = peak_month["month"].map(MONTH_NAMES)

peak_month[
    ["neighbourhood_number", "neighbourhood_name", "peak_month_name", "carts_installed"]
].head(20)

## 6. Spatial analysis

The peak-month map tests whether rollout timing appears geographically structured.

In [ ]:
geometry_df = api_query(
    CART_URL,
    {
        "$select": """
            neighbourhood_number,
            neighbourhood_name,
            geometry
        """,
        "$where": "geometry IS NOT NULL",
        "$group": """
            neighbourhood_number,
            neighbourhood_name,
            geometry
        """,
        "$limit": 1000
    }
)

geometry_df = geometry_df.dropna(subset=["geometry"]).copy()
geometry_df["neighbourhood_number"] = geometry_df["neighbourhood_number"].astype(str).str.strip()
geometry_df["geometry"] = geometry_df["geometry"].apply(wkt.loads)

edmonton_map = gpd.GeoDataFrame(
    geometry_df,
    geometry="geometry",
    crs="EPSG:4326"
)

edmonton_map.head()

In [ ]:
peak_for_map = peak_month.copy()
peak_for_map["neighbourhood_number"] = peak_for_map["neighbourhood_number"].astype(str).str.strip()

peak_map = edmonton_map.merge(
    peak_for_map[["neighbourhood_number", "peak_month_name", "carts_installed"]],
    on="neighbourhood_number",
    how="left",
    validate="1:1"
)

fig, ax = plt.subplots(figsize=(12, 12))
peak_map.plot(
    column="peak_month_name",
    categorical=True,
    legend=True,
    edgecolor="white",
    linewidth=0.3,
    ax=ax
)
ax.set_title("Peak 2021 cart rollout month by Edmonton neighbourhood")
ax.axis("off")
plt.show()

## 7. Build the core neighbourhood table

From this point forward, the analysis uses one row per neighbourhood.

In [ ]:
total_2021_by_neighbourhood = api_query(
    CART_URL,
    {
        "$select": """
            neighbourhood_number,
            neighbourhood_name,
            sum(number_of_carts) as carts_installed
        """,
        "$where": """
            install_date >= '2021-01-01T00:00:00'
            AND install_date < '2022-01-01T00:00:00'
        """,
        "$group": """
            neighbourhood_number,
            neighbourhood_name
        """,
        "$order": "sum(number_of_carts) DESC",
        "$limit": 5000
    }
)

total_2021_by_neighbourhood["neighbourhood_number"] = (
    total_2021_by_neighbourhood["neighbourhood_number"].astype(str).str.strip()
)
total_2021_by_neighbourhood["carts_installed"] = pd.to_numeric(
    total_2021_by_neighbourhood["carts_installed"], errors="coerce"
)

total_2021_by_neighbourhood.head(15)

## 8. Normalize by actual 2021 population

Using actual 2021 neighbourhood population is preferable to extrapolating the 2019 municipal census.

In [ ]:
population_2021 = api_query(POP_2021_URL, {"$limit": 5000})

population_2021["neighbourhood_number"] = (
    population_2021["neighbourhood_number"].astype(str).str.strip()
)
population_2021["total_population"] = pd.to_numeric(
    population_2021["total_population"], errors="coerce"
)

analysis_2021 = total_2021_by_neighbourhood.merge(
    population_2021[
        ["neighbourhood_number", "neighbourhood", "total_population", "ward"]
    ],
    on="neighbourhood_number",
    how="left",
    validate="1:1"
)

analysis_2021["carts_per_1000_residents"] = (
    analysis_2021["carts_installed"]
    / analysis_2021["total_population"]
    * 1000
)

print("Population join matched:",
      analysis_2021["total_population"].notna().sum(),
      "of",
      len(analysis_2021))

analysis_2021[
    ["neighbourhood_name", "carts_installed", "total_population", "carts_per_1000_residents"]
].sort_values("carts_per_1000_residents", ascending=False).head(20)

### Why population is not the final denominator

Residential carts are linked more directly to households or dwellings than to individual residents. A neighbourhood with smaller households can naturally have more carts per resident than a neighbourhood with larger households.

For operational benchmarking, occupied dwellings are more relevant.

## 9. Normalize by occupied private dwellings

The 2021 Census short-form dataset is in long format. `characteristics_code = 34` identifies the total occupied private dwellings measure used here.

In [ ]:
dwellings_2021 = api_query(
    CENSUS_2021_SHORT_URL,
    {
        "$select": """
            neighbourhood_number,
            neighbourhood_name,
            total_gender
        """,
        "$where": """
            characteristics_code = '34'
            AND neighbourhood_number IS NOT NULL
        """,
        "$limit": 5000
    }
)

dwellings_2021 = dwellings_2021.rename(
    columns={"total_gender": "occupied_dwellings_2021"}
)

dwellings_2021["neighbourhood_number"] = (
    dwellings_2021["neighbourhood_number"].astype(str).str.strip()
)
dwellings_2021["occupied_dwellings_2021"] = pd.to_numeric(
    dwellings_2021["occupied_dwellings_2021"], errors="coerce"
)

dwellings_2021.head()

In [ ]:
analysis_2021 = analysis_2021.merge(
    dwellings_2021[["neighbourhood_number", "occupied_dwellings_2021"]],
    on="neighbourhood_number",
    how="left",
    validate="1:1"
)

analysis_2021["carts_per_1000_dwellings"] = (
    analysis_2021["carts_installed"]
    / analysis_2021["occupied_dwellings_2021"]
    * 1000
)

print("Dwelling join matched:",
      analysis_2021["occupied_dwellings_2021"].notna().sum(),
      "of",
      len(analysis_2021))

analysis_2021[
    [
        "neighbourhood_name",
        "carts_installed",
        "occupied_dwellings_2021",
        "carts_per_1000_dwellings"
    ]
].sort_values("carts_per_1000_dwellings", ascending=False).head(20)

### Interpretation of the dwelling-based KPI

`2021-install-date carts per 1,000 occupied dwellings` is the most operationally useful measure in this notebook, but it is **not a literal coverage rate**.

Reasons:

- Census 2021 is a point-in-time snapshot.
- Cart installation dates span the full year.
- Fast-growing neighbourhoods can add homes after Census Day.
- Multiple cart types can exist for the same dwelling.

In [ ]:
map_dwellings = edmonton_map.merge(
    analysis_2021[["neighbourhood_number", "carts_per_1000_dwellings"]],
    on="neighbourhood_number",
    how="left",
    validate="1:1"
)

map_dwellings["quantile_bin"] = pd.qcut(
    map_dwellings["carts_per_1000_dwellings"],
    q=5,
    labels=False,
    duplicates="drop"
)

quantile_summary = (
    map_dwellings
    .dropna(subset=["quantile_bin"])
    .groupby("quantile_bin")["carts_per_1000_dwellings"]
    .agg(["min", "max", "count"])
)

labels = {
    idx: f"{row['min']:,.0f} – {row['max']:,.0f}"
    for idx, row in quantile_summary.iterrows()
}

map_dwellings["intensity_group"] = map_dwellings["quantile_bin"].map(labels)
ordered_labels = [labels[i] for i in sorted(labels)]

map_dwellings["intensity_group"] = pd.Categorical(
    map_dwellings["intensity_group"],
    categories=ordered_labels,
    ordered=True
)

fig, ax = plt.subplots(figsize=(12, 12))
map_dwellings.plot(
    column="intensity_group",
    categorical=True,
    legend=True,
    edgecolor="white",
    linewidth=0.3,
    ax=ax
)
ax.set_title(
    "2021 cart rollout intensity by Edmonton neighbourhood\n"
    "Carts with 2021 install dates per 1,000 occupied dwellings"
)
ax.axis("off")
plt.show()

## 10. Neighbourhood growth, 2019–2021

This section tests whether rapidly growing neighbourhoods tend to have higher rollout intensity.

**Caution:** 2019 values come from the Edmonton Municipal Census and 2021 values come from the Federal Census. The percentage change should therefore be treated as an **exploratory growth proxy**, not as a perfectly harmonized official time series.

In [ ]:
population_raw_2019 = api_query(
    POP_2019_URL,
    {
        "$select": """
            neighbourhood_number,
            neighbourhood_name,
            age_range,
            population
        """,
        "$limit": 50000
    }
)

population_raw_2019["population"] = pd.to_numeric(
    population_raw_2019["population"], errors="coerce"
)
population_raw_2019["neighbourhood_number"] = (
    population_raw_2019["neighbourhood_number"].astype(str).str.strip()
)

population_2019 = (
    population_raw_2019
    .groupby(["neighbourhood_number", "neighbourhood_name"], as_index=False)["population"]
    .sum()
    .rename(columns={"population": "population_2019"})
)

population_2019.head()

In [ ]:
growth_df = population_2019.merge(
    population_2021[["neighbourhood_number", "total_population"]],
    on="neighbourhood_number",
    how="inner",
    validate="1:1"
).rename(columns={"total_population": "population_2021"})

growth_df["population_change"] = (
    growth_df["population_2021"] - growth_df["population_2019"]
)

growth_df["population_growth_pct"] = (
    growth_df["population_change"]
    / growth_df["population_2019"]
    * 100
)

growth_df["population_growth_pct"] = (
    growth_df["population_growth_pct"]
    .replace([np.inf, -np.inf], np.nan)
)

growth_analysis = analysis_2021.merge(
    growth_df[
        [
            "neighbourhood_number",
            "population_2019",
            "population_2021",
            "population_change",
            "population_growth_pct"
        ]
    ],
    on="neighbourhood_number",
    how="left",
    validate="1:1"
)

growth_analysis[
    [
        "neighbourhood_name",
        "population_2019",
        "population_2021",
        "population_growth_pct",
        "occupied_dwellings_2021",
        "carts_installed",
        "carts_per_1000_dwellings"
    ]
].sort_values("carts_per_1000_dwellings", ascending=False).head(20)

To reduce unstable percentage growth caused by very small 2019 population bases, the correlation analysis below keeps neighbourhoods with at least 500 residents in 2019.

In [ ]:
growth_filtered = growth_analysis[
    (growth_analysis["population_2019"] >= 500)
    & growth_analysis["population_growth_pct"].notna()
    & growth_analysis["carts_per_1000_dwellings"].notna()
].copy()

pearson_corr = growth_filtered["population_growth_pct"].corr(
    growth_filtered["carts_per_1000_dwellings"],
    method="pearson"
)

spearman_corr = growth_filtered["population_growth_pct"].corr(
    growth_filtered["carts_per_1000_dwellings"],
    method="spearman"
)

absolute_corr = growth_filtered["population_change"].corr(
    growth_filtered["carts_installed"]
)

print("Pearson: growth % vs rollout intensity =", round(pearson_corr, 3))
print("Spearman: growth % vs rollout intensity =", round(spearman_corr, 3))
print("Absolute population increase vs carts =", round(absolute_corr, 3))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

ax.scatter(
    growth_filtered["population_growth_pct"],
    growth_filtered["carts_per_1000_dwellings"],
    s=growth_filtered["population_2021"] / 30,
    alpha=0.5
)

ax.axhline(
    growth_filtered["carts_per_1000_dwellings"].median(),
    linestyle="--",
    linewidth=1
)

ax.axvline(
    growth_filtered["population_growth_pct"].median(),
    linestyle="--",
    linewidth=1
)

ax.set_xlabel("Population change, 2019–2021 (%)")
ax.set_ylabel("2021-install-date carts per 1,000 occupied dwellings")
ax.set_title("Neighbourhood growth vs cart rollout intensity")

plt.tight_layout()
plt.show()

## 11. Four-segment neighbourhood classification

Median splits create four descriptive groups. This is a segmentation tool, **not a performance ranking**.

In [ ]:
growth_median = growth_filtered["population_growth_pct"].median()
rollout_median = growth_filtered["carts_per_1000_dwellings"].median()

def classify_neighbourhood(row):
    high_growth = row["population_growth_pct"] >= growth_median
    high_rollout = row["carts_per_1000_dwellings"] >= rollout_median

    if high_growth and high_rollout:
        return "High growth / High rollout"
    if high_growth and not high_rollout:
        return "High growth / Lower rollout"
    if not high_growth and high_rollout:
        return "Lower growth / High rollout"
    return "Lower growth / Lower rollout"

growth_filtered["segment"] = growth_filtered.apply(
    classify_neighbourhood,
    axis=1
)

growth_filtered["segment"].value_counts()

In [ ]:
segment_map = edmonton_map.merge(
    growth_filtered[["neighbourhood_number", "segment"]],
    on="neighbourhood_number",
    how="left",
    validate="1:1"
)

fig, ax = plt.subplots(figsize=(12, 12))
segment_map.plot(
    column="segment",
    categorical=True,
    legend=True,
    edgecolor="white",
    linewidth=0.3,
    ax=ax
)
ax.set_title("Neighbourhood growth and 2021 cart rollout intensity")
ax.axis("off")
plt.show()

## 12. Outlier analysis

The IQR method flags unusually high cart-to-dwelling intensity. These neighbourhoods should be investigated rather than automatically treated as errors.

In [ ]:
outlier_base = growth_analysis[
    growth_analysis["carts_per_1000_dwellings"].notna()
].copy()

q1 = outlier_base["carts_per_1000_dwellings"].quantile(0.25)
q3 = outlier_base["carts_per_1000_dwellings"].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

all_outliers = (
    outlier_base[
        outlier_base["carts_per_1000_dwellings"] > upper_bound
    ][
        [
            "neighbourhood_name",
            "population_2019",
            "population_2021",
            "population_growth_pct",
            "occupied_dwellings_2021",
            "carts_installed",
            "carts_per_1000_dwellings"
        ]
    ]
    .sort_values("carts_per_1000_dwellings", ascending=False)
)

print("Upper IQR outlier threshold:", round(upper_bound, 1))
all_outliers

## 13. Automated headline metrics

This cell produces a few concise metrics that can be copied into the executive summary after the notebook has been run successfully.

In [ ]:
year_2021_share = (
    yearly.loc[yearly["year"] == 2021, "carts_installed"].iloc[0]
    / yearly["carts_installed"].sum()
)

highest_intensity = (
    analysis_2021
    .dropna(subset=["carts_per_1000_dwellings"])
    .sort_values("carts_per_1000_dwellings", ascending=False)
    .iloc[0]
)

print(f"2021 share of carts in the installation-date distribution: {year_2021_share:.1%}")
print(f"January-August share of 2021 install-date carts: {jan_aug / total_2021:.1%}")
print(
    "Highest dwelling-normalized intensity:",
    highest_intensity["neighbourhood_name"],
    f"({highest_intensity['carts_per_1000_dwellings']:,.0f} per 1,000 occupied dwellings)"
)
print(f"Pearson growth-intensity correlation: {pearson_corr:.3f}")
print(f"Spearman growth-intensity correlation: {spearman_corr:.3f}")

## 14. Key findings

After running the notebook, the final narrative should focus on evidence from the outputs rather than on assumptions.

A strong final interpretation is likely to include these themes:

**2021 was the dominant install-date year.** The active-cart snapshot is heavily concentrated in 2021, making that year the natural focus of the rollout analysis.

**Rollout timing was concentrated and spatially uneven.** Monthly neighbourhood leaders and peak-month mapping indicate that different parts of Edmonton reached peak activity at different times.

**Normalization materially changes the story.** Absolute cart counts largely reflect neighbourhood size. Population helps, but occupied dwellings are the more relevant operational denominator.

**Fast growth explains some, but not all, high rollout intensity.** Newer neighbourhoods can show very high cart-to-dwelling ratios because the Census denominator is a point-in-time snapshot while installations continue through the year. At the same time, some established neighbourhoods also show high intensity.

**Extreme ratios require investigation, not automatic rejection.** They may reflect development timing, product mix, boundary effects, or data-quality issues.

## 15. Operational implications

The analysis supports several practical directions for deeper work:

**Planning denominators:** use occupied dwellings rather than total population when benchmarking residential cart infrastructure.

**Growth monitoring:** supplement Census counts with more frequently updated housing or development data in fast-growing neighbourhoods.

**Operational separation:** distinguish one-time city-wide rollout activity from ongoing cart demand generated by new residential development.

**Data quality:** review extreme neighbourhood ratios before using them for forecasting or resource allocation.

**Next data links:** development permits, dwelling completions, 311 service requests, missed collections, route data, and waste tonnage would allow the analysis to move from infrastructure rollout toward service performance and operational efficiency.

## 16. Limitations

- Cart Counts is an active-cart snapshot rather than a complete installation/removal history.
- Census reference dates and installation dates are not perfectly aligned.
- Multiple product types can exist for one dwelling.
- 2019 and 2021 population figures come from different census programs.
- Neighbourhood definitions can change over time.
- Correlation does not establish causation.
- Cart counts do not measure waste generation, collection cost, route efficiency, missed collections, or service quality.

### Recommended next phase

Link the cart analysis to **development permits or dwelling growth**, then test whether new residential construction predicts incremental cart demand after the 2021 city-wide rollout.